In [4]:
import numpy as np
from numpy import load
import tensorflow as tf
from numpy.random import randint
from numpy import zeros, ones
from scipy.signal import savgol_filter
from scipy.signal import butter, filtfilt
from scipy.ndimage import gaussian_filter1d


# generate points in latent space as input for the generator

def generate_latent_points(latent_dim, n_samples):

    x_input = gaussian_filter1d(np.random.randint(high=1.0, low=0.0,size=(latent_dim*n_samples)),4)
    z_input = x_input.reshape(n_samples, latent_dim)

    return z_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, real_sample, labels_input, latent_dim, n_samples):
    z_input = generate_latent_points(latent_dim, n_samples)
    images = generator.predict([real_sample,z_input, labels_input])
    y = zeros((n_samples, 1))
    return [images, labels_input], y

def generate_real_random(dataset, n_samples):
    images, labels = dataset
    ix = randint(0, images.shape[0], n_samples)
    X, labels = images[ix], labels[ix]
    y = ones((n_samples, 1))
    return [X, labels], y

def generate_real_samples(dataset, batch_id, n_samples):
    images, labels = dataset
    start = batch_id*n_samples
    end = start+n_samples
    X, labels = images[start:end], labels[start:end]
    y = ones((n_samples, 1))
    return [X, labels], y


In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import os

import random

def summarize_performance_fixed(reverse_dictionary, step,g_model,d_model ,dataset, n_samples=3,latent_dim=128,  savedir='weights_plots'):
    if not os.path.exists(savedir):
            os.makedirs(savedir)
    # select a sample of input images
    [X,labels],_ = generate_real_random(dataset, n_samples)
    # generate a batch of fake samples
    [X_fake, _], _ = generate_fake_samples(g_model,X, labels, latent_dim, n_samples)

    for url in X_fake:
                this_url_gen = ""
                for position in url:
                    this_index = np.argmax(position)
                    if this_index != 0:
                        this_url_gen += reverse_dictionary[this_index]

                print(this_url_gen)
    # save the generator model
    filename2 = savedir+'/'+'gmodel_%06d.h5' % (step+1)
    g_model.save(filename2)
    filename3 = savedir+'/'+'dmodel_%06d.h5' % (step+1)
    d_model.save(filename3)
    print('>Saved: %s and %s' % (filename2, filename3))


def to_csv(dr1_hist, dr2_hist, df1_hist, df2_hist, g_hist, gan_hist,savedir='dummy'):
    if not os.path.exists(savedir):
        os.makedirs(savedir)
    dr1 = np.array(dr1_hist)
    dr2 = np.array(dr2_hist)
    df1 = np.array(df1_hist)
    df2 = np.array(df2_hist)
    g = np.array(g_hist)
    gan = np.array(gan_hist)
    df = pd.DataFrame(data=(dr1,dr2,df1,df2,g,gan)).T
    df.columns=["dr1","dr2","df1","df2","g","gan"]
    filename = savedir+"/ecg-atk-loss.csv"
    df.to_csv(filename)

def plot_history(dr1_hist, dr2_hist, df1_hist, df2_hist, g_hist, gan_hist,savedir='dummy'):
    if not os.path.exists(savedir):
        os.makedirs(savedir)
    plt.plot(dr1_hist, label='dr1')
    plt.plot(dr2_hist, label='dr2')
    plt.plot(df1_hist, label='dfm1')
    plt.plot(df1_hist, label='dfm2')
    plt.plot(g_hist, label='g_loss')
    plt.plot(gan_hist, label='gan_loss')
    plt.legend()
    filename = savedir+'/plot_line_plot_loss.png'
    plt.savefig(filename)
    plt.close()
    print('Saved %s' % (filename))

In [7]:

import tensorflow as tf
from keras.layers import Layer, Reshape, Activation, Conv1D, Conv1DTranspose, Dropout
from keras.layers import Input, Add, Concatenate, Embedding,LeakyReLU,Dense, BatchNormalization, Flatten
from keras.optimizers import Adam
from keras.models import Model
from keras.initializers import RandomNormal
from functools import partial


# define the standalone discriminator model
def define_discriminator(in_shape=(200,67), n_classes=2):
    # weight initialization
    #init = RandomNormal(stddev=0.02)
    # image input
    in_image = Input(shape=(in_shape))
    # downsample to 14x14
    fe = Conv1D(16, 3, strides=1, padding='same')(in_image)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    fe = Conv1D(16, 3, strides=2, padding='same')(fe)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    fe = Dropout(0.2)(fe)
    # normal
    fe = Conv1D(32, 3, strides=1, padding='same')(fe)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    fe = Conv1D(32, 3, strides=2, padding='same')(fe)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    # downsample to 7x7
    fe = Conv1D(128, 3, strides=1, padding='same')(fe)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    fe = Conv1D(128, 3, strides=2, padding='same')(fe)
    fe = BatchNormalization()(fe)
    fe = LeakyReLU(alpha=0.2)(fe)
    fe = Dropout(0.2)(fe)
    # flatten feature maps
    fe = Flatten()(fe)
    dense_1 = Dense(256)(fe)
    dense_1 = LeakyReLU()(dense_1)
    dense_1 = Dense(64)(dense_1)
    # real/fake output
    out1 = Dense(1, activation='sigmoid')(fe)
    # class label output
    out2 = Dense(n_classes, activation='softmax')(fe)
    # define model
    model = Model(in_image, [out1, out2],name="Discriminator")
    # compile model
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss=['mse', 'categorical_crossentropy'], optimizer=opt)
    model.summary()
    return model
# define the standalone generator model
def define_generator(latent_dim=(50,),signal_shape=(200,67), label_shape=(2,)):
    # weight initialization
    #init = RandomNormal(stddev=0.02)
    depth = 4 #32
    dropout = 0.25
    dim = signal_shape[0] #

    # signal_input
    in_signal = Input(shape=signal_shape)
    si = in_signal
    #si = Reshape((280,1))(in_signal)

    # label input
    in_label = Input(shape=label_shape)
    # embedding for categorical input
    li = Embedding(2, 50)(in_label)
    # linear multiplication
    n_nodes = 200 * 1
    li = Dense(n_nodes)(li)
    # reshape to additional channel
    li = Reshape((200,2))(li)

    # noise  input
    in_lat = Input(shape=latent_dim)
    lat = Reshape((1,50))(in_lat)
    # foundation for 7x7 image
    n_nodes = dim*depth
    gen = Dense(n_nodes)(lat)
    gen = LeakyReLU(alpha=0.2)(gen)
    gen = Reshape((dim, depth))(gen)
    # merge image gen and label input
    merge = Concatenate()([gen, li,si]) #gen=200,32 x li=200,2 x si=200,67 ## Uncomment this
    #merge = Concatenate()([gen, li]) #gen=280,32 li=280,5


    gen = Conv1D(32, 3, strides=1, padding='same')(merge)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1D(32, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1D(64, 3, strides=1, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1D(64, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1D(128, 3, strides=1, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1D(128, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1DTranspose(128, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1DTranspose(64, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)

    gen = Conv1DTranspose(32, 3, strides=2, padding='same')(gen)
    gen = BatchNormalization()(gen)
    gen = LeakyReLU(alpha=0.2)(gen)


    #gen = Reshape((200,67))(gen)

    gen = Conv1D(67, 3, strides=1, padding='same')(gen)
    out_layer = Activation('sigmoid')(gen)

    model = Model([in_signal,in_lat, in_label], out_layer,name="Generator")
    #model = Model([in_lat, in_label], out_layer,name="Generator")
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='mse', optimizer=opt)
    model.summary()
    return model
def define_gan(g_model, d_model,latent_dim=(200,67), signal_shape=(200,67),label_shape=(2,)):
    #in_signal = Input(shape=signal_shape)
    #in_label = Input(shape=label_shape)
    #in_lat = Input(shape=latent_dim)
    # make weights in the discriminator not trainable
    d_model.trainable = False
    # connect the outputs of the generator to the inputs of the discriminator
    [out1,out2] = d_model(g_model.output)
    # define gan model as taking noise and label and outputting real/fake and label outputs
    #model = Model(g_model.input, gan_output)
    model = Model([g_model.input[0],g_model.input[1],g_model.input[2]],[out1,out2])
    #model = Model([g_model.input[0],g_model.input[1]],[out1,out2])
    #model = Model([in_signal,in_lat, in_label],[out1,out2])
    # compile model
    opt = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss=['mse', 'categorical_crossentropy'], optimizer=opt,loss_weights=[1,10])
    model.summary()
    return model

In [8]:
import pandas as pd
import keras
from keras.utils import to_categorical
import numpy as np
import string
import argparse

def tokenizer(alphabet):
    dictionary = {char: i+1 for i, char in enumerate(alphabet)}
    reverse_dictionary = {i+1: char for i, char in enumerate(alphabet)}
    return dictionary, reverse_dictionary

def process_line(line, alphabet, dictionary, url_length):
    line = line.lower()
    if len(set(line) - set(alphabet)) == 0 and len(line) < url_length:
        this_sample = np.zeros((url_length, len(alphabet) + 1))
        for i, char in enumerate(line):
            this_sample[i, 0] = 0.0
            this_sample[i, dictionary[char]] = 1.0
        return this_sample
    else:
        return None

def data_generator(data, alphabet, dictionary, url_length, samples):
    processed_data = []
    for i in range(len(data)):
        line = data['URL'][i]
        this_sample = process_line(line, alphabet, dictionary, url_length)
        if this_sample is not None:
            processed_data.append(this_sample)
        if len(processed_data) == samples:
            break
    return np.array(processed_data)

def data_npz(good, bad, alphabet, dictionary, samples, url_length, npz_filename='phishing.npz'):
    good_data = data_generator(good, alphabet, dictionary, url_length, samples // 2)
    bad_data = data_generator(bad, alphabet, dictionary, url_length, samples // 2)
    print("Good data shape:", good_data.shape)
    print("Bad data shape:", bad_data.shape)

    x_train_len = int(samples * 0.8) // 2
    x_train = np.concatenate((good_data[:x_train_len], bad_data[:x_train_len]), axis=0)
    x_test = np.concatenate((good_data[x_train_len:samples // 2], bad_data[x_train_len:samples // 2]), axis=0)

    y_train = np.concatenate((np.ones((x_train_len, 1)), np.zeros((x_train_len, 1))), axis=0)
    y_train_cat = to_categorical(y_train)
    y_test = np.concatenate((np.ones((samples // 2 - x_train_len, 1)), np.zeros((samples // 2 - x_train_len, 1))), axis=0)
    y_test_cat = to_categorical(y_test)

    np.savez_compressed(npz_filename, X_train=x_train, X_test=x_test, y_train=y_train_cat, y_test=y_test_cat)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--url_length', type=int, default=200)
    parser.add_argument('--npz_filename', type=str, default='phishing.npz')
    parser.add_argument('--n_samples', type=int, default=50000, help='number of good and bad samples.')
    args, _ = parser.parse_known_args()

    alphabet = string.ascii_lowercase + string.digits + "!#$%&()*+,-./:;<=>?@[\\]^_`{|}~"
    dictionary, _ = tokenizer(alphabet)

    df = pd.read_csv('/content/phishing_site_urls.csv')
    good = df[df['Label'] == 'good']
    bad = df[df['Label'] == 'bad']
    good.reset_index(drop=True, inplace=True)
    bad.reset_index(drop=True, inplace=True)

    data_npz(good, bad, alphabet, dictionary, args.n_samples, args.url_length, npz_filename=args.npz_filename)


Good data shape: (25000, 200, 67)
Bad data shape: (25000, 200, 67)


In [ ]:
import numpy as np
import os
import string
import time
import argparse
import keras.backend as K
from sklearn.utils import shuffle
import numpy as np
import tensorflow as tf

# Set seed for reproducibility
seed_value = 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)


def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=20, n_batch=32, savedir="dummy"):
    bat_per_epo = int(dataset[0].shape[0] / n_batch)
    print('batch per epoch: %d' % bat_per_epo)
    n_steps = bat_per_epo * n_epochs
    print('number of steps: %d' % n_steps)
    dr1_hist, dr2_hist, df1_hist, df2_hist = list(), list(), list(), list()
    g_hist, gan_hist = list(), list()

    alphabet = string.ascii_lowercase + string.digits + "!#$%&()*+,-./:;<=>?@[\\]^_`{|}~"
    reverse_dictionary = {i+1: c for i, c in enumerate(alphabet)}
    b = 0
    start_time = time.time()
    for e in range(n_epochs):
        for i in range(bat_per_epo):
            for j in range(2):
                d_model.trainable = True
                g_model.trainable = False
                [X_real, labels_real], y_real = generate_real_samples(dataset, i, n_batch)
                _, d_r1, d_r2 = d_model.train_on_batch(X_real, [y_real, labels_real])
                [X_fake, labels_fake], y_fake = generate_fake_samples(g_model, X_real, labels_real, latent_dim, n_batch)
                _, d_f1, d_f2 = d_model.train_on_batch(X_fake, [y_fake, labels_fake])
            d_model.trainable = False
            g_model.trainable = True
            z_input = generate_latent_points(latent_dim, n_batch)
            [X_real, labels_real], y_real = generate_real_samples(dataset, i, n_batch)
            g_loss = g_model.train_on_batch([X_real, z_input, labels_real], X_real)
            gan_loss, _, _ = gan_model.train_on_batch([X_real, z_input, labels_real], [y_real, labels_real])
            print('>%d, dr[%.3f,%.3f], df[%.3f,%.3f], g[%.3f], gan[%.3f]' % (i+1, d_r1, d_r2, d_f1, d_f2, g_loss, gan_loss))
            dr1_hist.append(d_r1)
            dr2_hist.append(d_r2)
            df1_hist.append(d_f1)
            df2_hist.append(d_f2)
            g_hist.append(g_loss)
            gan_hist.append(gan_loss)

        summarize_performance_fixed(reverse_dictionary, b, g_model, d_model, dataset, 3, latent_dim, savedir=savedir)
        b += 1
        per_epoch_time = time.time()
        total_per_epoch_time = (per_epoch_time - start_time) / 3600.0
        print(total_per_epoch_time)

    plot_history(dr1_hist, dr2_hist, df1_hist, df2_hist, g_hist, gan_hist, savedir=savedir)
    to_csv(dr1_hist, dr2_hist, df1_hist, df2_hist, g_hist, gan_hist, savedir=savedir)

def main(args):
    K.clear_session()
    if not os.path.exists(args.savedir):
        os.makedirs(args.savedir)

    discriminator = define_discriminator()
    generator = define_generator(args.latent_dim)

    if args.resume_training == 'yes':
        discriminator.load_weights(args.weight_name_dis)
        generator.load_weights(args.weight_name_gen)

    gan_model = define_gan(generator, discriminator)
    data = np.load(args.npz_file)
    dataset = shuffle(data['X_train'], data['y_train'])
    train(generator, discriminator, gan_model, dataset, latent_dim=args.latent_dim, n_epochs=args.epochs, n_batch=args.batch_size, savedir=args.savedir)

if __name__ == "__main__":
    import sys
    if any(arg.startswith("-f") or arg.startswith("--f") for arg in sys.argv):
        print("Running in Jupyter Notebook")
        class Args:
            epochs = 149
            batch_size = 70
            npz_file = 'phishing.npz'
            latent_dim = 50
            savedir = 'PhishGan'
            resume_training = 'no'
            weight_name_dis = None
            weight_name_gen = None
        args = Args()
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--epochs', type=int, default=149)
        parser.add_argument('--batch_size', type=int, default=70)
        parser.add_argument('--npz_file', type=str, default='phishing.npz', help='path/to/npz/file')
        parser.add_argument('--latent_dim', type=int, default=50)
        parser.add_argument('--savedir', type=str, required=False, help='path/to/save_directory', default='PhishGan')
        parser.add_argument('--resume_training', type=str, required=False, default='no', choices=['yes', 'no'])
        parser.add_argument('--weight_name_dis', type=str, help='path/to/discriminator/weight/.h5 file', required=False)
        parser.add_argument('--weight_name_gen', type=str, help='path/to/generator/weight/.h5 file', required=False)
        args = parser.parse_args()

    main(args)


Running in Jupyter Notebook
Model: "Discriminator"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 200, 67)]            0         []                            
                                                                                                  
 conv1d (Conv1D)             (None, 200, 16)              3232      ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 200, 16)              64        ['conv1d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 leaky_re_lu (LeakyReLU)     (None, 200, 16)              

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Streaming output truncated to the last 5000 lines.
>53, dr[0.000,0.009], df[0.000,0.000], g[0.000], gan[0.033]
3/3 [==============================] - 0s 10ms/step
>54, dr[0.000,0.004], df[0.000,0.000], g[0.000], gan[0.029]
3/3 [==============================] - 0s 10ms/step
>55, dr[0.000,0.011], df[0.000,0.000], g[0.000], gan[0.003]
3/3 [==============================] - 0s 10ms/step
>56, dr[0.000,0.000], df[0.000,0.000], g[0.000], gan[0.002]
3/3 [==============================] - 0s 10ms/step
>57, dr[0.000,0.007], df[0.000,0.000], g[0.000], gan[0.006]
3/3 [==============================] - 0s 11ms/step
>58, dr[0.000,0.000], df[0.000,0.000], g[0.000], gan[0.002]
3/3 [==============================] - 0s 9ms/step
>59, dr[0.000,0.002], df[0.000,0.000], g[0.000], gan[0.004]
3/3 [==============================] - 0s 8ms/step
>60, dr[0.000,0.028], df[0.000,0.000], g[0.000], gan[0.004]
3/3 [==============================] - 0s 9ms/step
>61, dr[0.000,0.019], df[0.000,0.000], g[0.000], gan[0.0

In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, confusion_matrix
from skimage.metrics import normalized_root_mse, structural_similarity as ssim
import os
import argparse
import keras.backend as K
import tensorflow as tf
from keras.models import load_model

# Set seed for reproducibility
seed_value = 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

def evaluate_similarity(g_model, dataset, latent_dim):
    X_test, y_test = dataset
    # Generate fake samples
    z_input = generate_latent_points(latent_dim, X_test.shape[0])
    y_labels = y_test  # Use real labels for generating fake samples
    X_fake = g_model.predict([X_test, z_input, y_labels])

    # Flatten the arrays for mean_squared_error
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    X_fake_flat = X_fake.reshape(X_fake.shape[0], -1)

    # Calculate MSE, NRMSE, and SSIM between real and fake samples
    mse_value = mean_squared_error(X_test_flat, X_fake_flat)
    nrmse_value = normalized_root_mse(X_test, X_fake, normalization='euclidean')
    ssim_value = ssim(X_test, X_fake, data_range=X_test.max() - X_test.min(), channel_axis=-1)

    # Print metrics
    print("\nGenerator’s Performance: Similarity between real and adversarial Phishing URLs")
    print(f"MSE: {mse_value}")
    print(f"NRMSE: {nrmse_value}")
    print(f"SSIM: {ssim_value}")

def evaluate_classification_performance(d_model, dataset):
    X_test, y_test = dataset
    y_true = np.argmax(y_test, axis=1)

    # Get predictions from discriminator
    _, y_pred_labels_real = d_model.predict(X_test)
    y_pred_real_labels = np.argmax(y_pred_labels_real, axis=1)

    # Compute metrics for real samples
    accuracy_real = accuracy_score(y_true, y_pred_real_labels)
    precision_real = precision_score(y_true, y_pred_real_labels)
    recall_real = recall_score(y_true, y_pred_real_labels)
    f1_real = f1_score(y_true, y_pred_real_labels)


    # Print metrics
    print("\nPhishing URL Classification Performance")
    print(f"Accuracy: {accuracy_real}")
    print(f"Sensitivity (Recall): {recall_real}")
    print(f"Precision: {precision_real}")
    print(f"F1 Score: {f1_real}")


def evaluate_adversarial_detection(d_model, g_model, X_test, y_test, latent_dim):
    # Generate fake samples
    z_input = generate_latent_points(latent_dim, X_test.shape[0])
    y_labels = y_test
    X_fake = g_model.predict([X_test, z_input, y_labels])

    # Combine real and fake samples
    X_combined = np.concatenate((X_test, X_fake), axis=0)
    y_combined = np.concatenate((y_test, np.zeros_like(y_test)), axis=0)

    # Get predictions from discriminator
    _, y_pred_labels = d_model.predict(X_combined)
    y_pred = np.argmax(y_pred_labels, axis=1)

    y_combined_labels = np.argmax(y_combined, axis=1)

    # Compute metrics
    accuracy = accuracy_score(y_combined_labels, y_pred)
    precision = precision_score(y_combined_labels, y_pred)
    recall = recall_score(y_combined_labels, y_pred)
    f1 = f1_score(y_combined_labels, y_pred)


    # Calculate specificity
    tn, fp, fn, tp = confusion_matrix(y_combined_labels, y_pred).ravel()
    specificity = tn / (tn + fp)

    print("\nAdversarial Example Detection: Discriminator’s performance for detecting adversarial examples")
    print(f"Accuracy: {accuracy}")
    print(f"Sensitivity (Recall): {recall}")
    print(f"Specificity: {specificity}")
    print(f"F1 Score: {f1}")


if __name__ == "__main__":
    # Argument parsing
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=149)
    parser.add_argument('--batch_size', type=int, default=70)
    parser.add_argument('--npz_file', type=str, default='phishing.npz', )
    parser.add_argument('--latent_dim', type=int, default=50)
    parser.add_argument('--savedir', type=str, required=False, default='PhishGan')
    parser.add_argument('--resume_training', type=str, required=False, default='no', choices=['yes', 'no'])
    parser.add_argument('--weight_name_dis', type=str, required=False)
    parser.add_argument('--weight_name_gen', type=str, required=False)

    import sys
    if 'ipykernel' in sys.modules:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    K.clear_session()
    if not os.path.exists(args.savedir):
        os.makedirs(args.savedir)

    # Get the latest model files from the savedir
    g_model_files = sorted([f for f in os.listdir(args.savedir) if f.startswith('Copy of gmodel_') and f.endswith('.h5')])
    d_model_files = sorted([f for f in os.listdir(args.savedir) if f.startswith('Copy of dmodel_') and f.endswith('.h5')])

    if g_model_files and d_model_files:
        final_g_model_path = os.path.join(args.savedir, g_model_files[148])
        final_d_model_path = os.path.join(args.savedir, d_model_files[148])
        print(f'Loading generator model from: {final_g_model_path}')
        print(f'Loading discriminator model from: {final_d_model_path}')

        generator = load_model(final_g_model_path)
        discriminator = load_model(final_d_model_path)
    else:
        print("No model files found.")
        exit()

    # Load test dataset
    data = np.load(args.npz_file)
    X_test = data['X_test']
    y_test = data['y_test']

    # Evaluate the model
    evaluate_similarity(generator, (X_test, y_test), latent_dim=args.latent_dim)
    evaluate_classification_performance(discriminator, (X_test, y_test))
    evaluate_adversarial_detection(discriminator, generator, X_test, y_test, latent_dim=args.latent_dim)


Loading generator model from: PhishGan/Copy of gmodel_000149.h5
Loading discriminator model from: PhishGan/Copy of dmodel_000149.h5
313/313 [==============================] - 4s 12ms/step

Generator’s Performance: Similarity between real and adversarial Phishing URLs
MSE: 0.000322316512239019
NRMSE: 0.2875271048587865
SSIM: 0.9904722356064107
313/313 [==============================] - 2s 6ms/step

Phishing URL Classification Performance
Accuracy: 0.9536
Sensitivity (Recall): 0.9652
Precision: 0.9433150899139953
F1 Score: 0.9541320680110714
625/625 [==============================] - 4s 6ms/step

Adversarial Example Detection: Discriminator’s performance for detecting adversarial examples
Accuracy: 0.7191
Sensitivity (Recall): 0.9652
Specificity: 0.6370666666666667
F1 Score: 0.6320890635232482
